# **Removing Duplicate/Inadequete Rows**

In [2]:
import pandas as pd

df = pd.read_csv("scraped_jobs.csv")
df = df.dropna(subset=["description"])

initial_rows = len(df)
print("Initial dataset size:",initial_rows)

df = df.drop_duplicates(
    subset=["role", "description"],
    keep="first"
)



import re

def has_valid_words(text, min_words=10):
    words = re.findall(r"[a-zA-Z]{2,}", text)
    return len(words) >= min_words

df = df[df["description"].apply(has_valid_words)]

len(df)



df["desc_length"] = df["description"].str.len()
avg_length = df["desc_length"].mean()
MIN_LENGTH = avg_length * 0.4
df = df[df["desc_length"] >= MIN_LENGTH]

def valid_job_title(title):
    if pd.isna(title):
        return False
    title = title.lower()
    if len(title) < 3:
        return False
    if not re.search(r"[a-zA-Z]", title):
        return False
    return True

df = df[df["role"].apply(valid_job_title)]

print(f"Final dataset size: {len(df)} job postings")



Initial dataset size: 954
Final dataset size: 764 job postings


# **Tokenization & Lemmatization**

In [3]:
import re

def clean_text(text):
    text = text.lower()
    text = re.sub(r"<.*?>", " ", text)      # remove html
    text = re.sub(r"[^a-zA-Z\s]", " ", text) # remove punctuation/numbers
    text = re.sub(r"\s+", " ", text).strip()
    return text

df["clean_description"] = df["description"].apply(clean_text)

import spacy
nlp = spacy.load("en_core_web_sm")

def lemmatize(text):
    doc = nlp(text)
    return [token.lemma_ for token in doc
            if not token.is_stop and token.is_alpha]

df["tokens"] = df["clean_description"].apply(lemmatize)


# **Role Normalization**

In [4]:
import re

def clean_job_title(title):
    title = title.lower()
    title = re.sub(r"@.*", "", title)                # remove company
    title = re.sub(r"\(.*?\)", "", title)            # remove brackets
    title = re.sub(r"[-/|]", " ", title)              # separators
    title = re.sub(r"\b(senior|jr|junior|lead|tech lead|principal)\b", "", title)
    title = re.sub(r"\s+", " ", title).strip()
    return title

df["clean_title"] = df["role"].apply(clean_job_title)

ROLE_KEYWORDS = {
    # --- Engineering & Development ---
    "software engineer": ["software engineer", "software developer", "swe", "developer", "application developer", "systems programmer"],
    "web developer": ["web developer", "frontend developer", "backend developer", "full stack developer", "ui developer", "javascript developer"],
    "mobile developer": ["mobile developer", "android developer", "ios developer", "react native developer", "flutter developer", "swift developer"],
    "game developer": ["game developer", "unity developer", "unreal engine developer", "game programmer", "graphics engineer"],
    "embedded engineer": ["embedded engineer", "firmware engineer", "iot developer", "hardware engineer", "systems engineer"],

    # --- Data & AI ---
    "data scientist": ["data scientist", "machine learning engineer", "ml engineer", "ai engineer", "nlp engineer", "computer vision engineer", "deep learning engineer"],
    "data analyst": ["data analyst", "business analyst", "bi analyst", "product analyst", "data visualizer", "tableau developer"],
    "data engineer": ["data engineer", "etl developer", "big data engineer", "data architect", "analytics engineer"],
    "database administrator": ["dba", "database administrator", "database engineer", "sql developer"],

    # --- Infrastructure, Cloud & DevOps ---
    "devops engineer": ["devops engineer", "site reliability engineer", "sre", "platform engineer", "automation engineer", "build engineer"],
    "cloud engineer": ["cloud engineer", "cloud architect", "aws architect", "azure engineer", "gcp architect", "cloud consultant"],
    "network engineer": ["network engineer", "network architect", "systems administrator", "sysadmin", "infrastructure engineer"],

    # --- Security ---
    "cybersecurity engineer": ["security engineer", "cybersecurity", "information security", "soc analyst", "pentester", "ethical hacker", "application security engineer", "security architect"],
    "compliance officer": ["it compliance", "grc analyst", "it auditor", "privacy engineer"],

    # --- Quality & Testing ---
    "qa engineer": ["qa engineer", "quality assurance", "software tester", "automation tester", "sdet", "manual tester", "performance engineer"],

    # --- Product & Management ---
    "product manager": ["product manager", "pm", "technical product manager", "product owner"],
    "project manager": ["project manager", "it project manager", "scrum master", "agile coach", "delivery manager"],
    "it manager": ["it manager", "cto", "cio", "engineering manager", "vpe", "it director"],

    # --- Design & UX ---
    "ux designer": ["ux designer", "ui designer", "product designer", "user researcher", "interaction designer", "ux writer"],

    # --- Specialized Platforms & ERP ---
    "salesforce developer": ["salesforce developer", "sfdc developer", "salesforce admin", "crm developer"],
    "erp consultant": ["sap consultant", "oracle functional consultant", "dynamics 365 developer", "erp analyst"],
    "service management": ["itsm manager", "servicenow developer", "itil consultant"]
}

def normalize_role(title):
    for canonical_role, keywords in ROLE_KEYWORDS.items():
        for kw in keywords:
            if kw in title:
                return canonical_role
    return "other"

df["normalized_role"] = df["clean_title"].apply(normalize_role)
df

,role,description,desc_length,clean_description,tokens,clean_title,normalized_role
0,Software Developer,Software Developer\nDiligent Consulting Group ...,1513,software developer diligent consulting group p...,"[software, developer, diligent, consulting, gr...",software developer,software engineer
1,Software Engineer,Software Engineer\nHayleys\n| Â 2024-06-01\nSi...,1400,software engineer hayleys similar jobs apply a...,"[software, engineer, hayley, similar, job, app...",software engineer,software engineer
2,Senior Software Engineer - Next.js/React.js @i...,Senior Software Engineer - Next.js/React.js @i...,2340,senior software engineer next js react js ilab...,"[senior, software, engineer, js, react, js, il...",software engineer next.js react.js,software engineer
3,Senior Tech Lead - Software Engineering,Senior Tech Lead - Software Engineering\nDialo...,1956,senior tech lead software engineering dialog s...,"[senior, tech, lead, software, engineering, di...",software engineering,software engineer
4,Senior Software Engineer,Senior Software Engineer\nHayleys\n| Â 2024-11...,1346,senior software engineer hayleys similar jobs ...,"[senior, software, engineer, hayley, similar, ...",software engineer,software engineer
...,...,...,...,...,...,...,...
944,Graphic Designer,CA .\nF PO) HOME ABOUT US PROFILE CONTACT US\n...,1529,ca f po home about us profile contact us ss in...,"[f, po, home, profile, contact, ss, industrial...",graphic designer,other
948,Application Support Associate,K\nZeeks Lab\nAre you passionate about\nTechno...,2188,k zeeks lab are you passionate about technolog...,"[k, zeeks, lab, passionate, technology, custom...",application support associate,other
949,UI-UX Developer,"At eBEYONDS, an International eBusiness & Digi...",1601,at ebeyonds an international ebusiness digital...,"[ebeyond, international, ebusiness, digital, m...",ui ux developer,software engineer
953,Consultant (SAP FICO) (1),"John Keells Information Technology (Pvt} Ltd, ...",2366,john keells information technology pvt ltd the...,"[john, keells, information, technology, pvt, l...",consultant,other


In [5]:
df.to_csv('final_job_postings.csv', index=False)

The cleaned and normalized job postings have been saved to `final_job_postings.csv`.

In [6]:
other_roles_count = len(df[df['normalized_role'] == 'other'])
print(f"Number of rows with 'other' in normalized_role: {other_roles_count}")

Number of rows with 'other' in normalized_role: 118


In [7]:
import nltk
from nltk.corpus import stopwords

nltk.download('stopwords')

stop_words = set(stopwords.words("english"))

def clean_description(text):
    text = text.lower()
    text = re.sub(r"[^a-zA-Z ]", " ", text)
    words = text.split()
    words = [w for w in words if w not in stop_words and len(w) > 2]
    return " ".join(words)

df["clean_description"] = df["description"].apply(clean_description)

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


In [8]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(
    max_features=3000,
    ngram_range=(1, 2)   # VERY important for skills
)

tfidf_matrix = vectorizer.fit_transform(df["clean_description"])



In [9]:
import numpy as np
import pandas as pd

feature_names = vectorizer.get_feature_names_out()

role_skill_rankings = {}

for role in df["normalized_role"].unique():
    role_df = df[df["normalized_role"] == role]

    #if len(role_df) < 5:
        #continue  # avoid noise

    role_tfidf = vectorizer.transform(role_df["clean_description"])
    mean_scores = np.mean(role_tfidf.toarray(), axis=0)

    role_skills = pd.DataFrame({
        "skill": feature_names,
        "score": mean_scores
    }).sort_values(by="score", ascending=False)

    role_skill_rankings[role] = role_skills


In [10]:
IT_SKILLS = [

    # Programming Languages
    "python", "java", "c", "c++", "c#", "javascript", "typescript", "go", "rust",
    "php", "ruby", "swift", "kotlin", "r", "matlab", "scala", "perl", "bash",
    "powershell", "objective-c", "groovy", "dart", "lua", "haskell",

    # Web Development
    "html", "css", "sass", "less", "bootstrap", "tailwind css",
    "react", "angular", "vue", "next.js", "nuxt.js", "svelte",
    "node.js", "express.js", "nestjs",
    "django", "flask", "fastapi",
    "spring", "spring boot",
    "laravel", "codeigniter",
    "asp.net", "asp.net core",
    "graphql", "rest api", "soap",

    # Databases
    "mysql", "postgresql", "oracle", "sql server", "sqlite",
    "mongodb", "cassandra", "couchdb", "redis", "dynamodb",
    "firebase", "neo4j", "elasticsearch",
    "nosql", "sql", "pl/sql",

    # Cloud & DevOps
    "aws", "azure", "google cloud", "gcp",
    "ec2", "s3", "lambda", "cloudformation",
    "docker", "kubernetes", "helm",
    "terraform", "ansible", "chef", "puppet",
    "jenkins", "gitlab ci", "github actions", "circleci",
    "linux", "unix",
    "nginx", "apache",
    "devops", "site reliability engineering", "sre",

    # Data Science & Machine Learning
    "machine learning", "deep learning", "artificial intelligence",
    "natural language processing", "nlp", "computer vision",
    "data science", "data analysis", "data engineering",
    "pandas", "numpy", "scipy", "scikit-learn",
    "tensorflow", "keras", "pytorch",
    "xgboost", "lightgbm",
    "opencv",
    "statistics", "linear regression", "logistic regression",
    "clustering", "classification", "time series",

    # Big Data
    "hadoop", "spark", "pyspark", "kafka", "flink",
    "hive", "pig", "hbase", "airflow",
    "data warehousing", "etl",

    # Mobile Development
    "android", "ios",
    "react native", "flutter", "xamarin",
    "android studio", "xcode",

    # Cybersecurity
    "cybersecurity", "information security",
    "penetration testing", "ethical hacking",
    "network security", "application security",
    "cryptography", "siem", "soc",
    "firewalls", "ids", "ips",
    "owasp", "iam",

    # Networking
    "tcp/ip", "udp", "dns", "dhcp",
    "http", "https",
    "routing", "switching",
    "vpn", "lan", "wan",
    "ccna", "ccnp",

    # Operating Systems
    "windows", "linux", "macos",
    "red hat", "ubuntu", "debian", "centos",

    # Software Engineering
    "object oriented programming", "oop",
    "design patterns",
    "clean code", "solid principles",
    "data structures", "algorithms",
    "microservices", "monolithic architecture",
    "event driven architecture",

    # Testing & QA
    "unit testing", "integration testing", "system testing",
    "selenium", "cypress", "playwright",
    "junit", "pytest", "testng",
    "automation testing", "manual testing",

    # Version Control & Tools
    "git", "github", "gitlab", "bitbucket",
    "jira", "confluence",
    "postman", "swagger",

    # UI / UX
    "figma", "adobe xd", "sketch",
    "ui design", "ux design",
    "wireframing", "prototyping",

    # ERP / CRM / Enterprise
    "sap", "oracle erp", "salesforce",
    "workday", "servicenow",

    # Methodologies
    "agile", "scrum", "kanban",
    "waterfall", "devsecops",

    # Misc / Emerging
    "blockchain", "web3", "smart contracts",
    "solidity",
    "internet of things", "iot",
    "robotic process automation", "rpa",
    "computer graphics", "game development",
    "unity", "unreal engine"

]

def filter_skills(df):
    return df[df["skill"].isin(IT_SKILLS)]

filtered_role_skills = {
    role: filter_skills(skills)
    for role, skills in role_skill_rankings.items()
}


In [11]:
print(df['normalized_role'].unique())

desired_roles = [
    "software engineer",
    "qa engineer",
    "cybersecurity engineer",
    "data engineer",
    "data scientist",
    "database administrator",
    "devops engineer",
    "embedded engineer",
    "data analyst"
]

for role in desired_roles:
    if role in filtered_role_skills:
        print(f"\nSkills for {role}:")
        print(filtered_role_skills[role].head(10)) # Print top 10 skills
    else:
        print(f"\nSkills for '{role}' not available (likely due to insufficient data).")

['software engineer' 'qa engineer' 'other' 'product manager' 'it manager'
 'cybersecurity engineer' 'data engineer' 'data scientist'
 'database administrator' 'devops engineer' 'project manager'
 'embedded engineer' 'data analyst']

Skills for software engineer:
                 skill     score
2195             react  0.022384
1459        javascript  0.018972
227                aws  0.014396
2154            python  0.013578
1456              java  0.013029
1621  machine learning  0.012493
608                css  0.012122
2798        typescript  0.011586
2522               sql  0.011166
80               agile  0.010348

Skills for qa engineer:
           skill     score
80         agile  0.037583
1463        jira  0.036149
2368  salesforce  0.029615
2394       scrum  0.022701
2522         sql  0.018418
1456        java  0.016430
747       devops  0.014845
1459  javascript  0.013480
1462     jenkins  0.011959
2519      spring  0.011717

Skills for cybersecurity engineer:
                

In [12]:
import sys

# Install spacy and spacy-transformers
!{sys.executable} -m pip install spacy spacy-transformers

# Download the large English model
!{sys.executable} -m spacy download en_core_web_trf

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 795.8/795.8 kB 19.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 313.4/313.4 kB 17.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 58.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 32.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 60.5 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.5.0
    Uninstalling huggingface_hub-1.5.0:
      Successfully uninstalled huggingface_hub-1.5.0
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.22.2
    Uninstalling tokenizers-0.22.2:
      Successfully uninstalled tokenizers-0.22.2
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled trans

In [13]:
SKILLS = [
    # Programming Languages
    "python", "java", "c", "c++", "c#", "javascript", "typescript", "go", "rust",
    "php", "ruby", "swift", "kotlin", "r", "matlab", "scala", "perl", "bash",
    "powershell", "objective-c", "groovy", "dart", "lua", "haskell",

    # Web Development
    "html", "css", "sass", "less", "bootstrap", "tailwind css",
    "react", "angular", "vue", "next.js", "nuxt.js", "svelte",
    "node.js", "express.js", "nestjs",
    "django", "flask", "fastapi",
    "spring", "spring boot",
    "laravel", "codeigniter",
    "asp.net", "asp.net core",
    "graphql", "rest api", "soap",

    # Databases
    "mysql", "postgresql", "oracle", "sql server", "sqlite",
    "mongodb", "cassandra", "couchdb", "redis", "dynamodb",
    "firebase", "neo4j", "elasticsearch",
    "nosql", "sql", "pl/sql",

    # Cloud & DevOps
    "aws", "azure", "google cloud", "gcp",
    "ec2", "s3", "lambda", "cloudformation",
    "docker", "kubernetes", "helm",
    "terraform", "ansible", "chef", "puppet",
    "jenkins", "gitlab ci", "github actions", "circleci",
    "linux", "unix",
    "nginx", "apache",
    "devops", "site reliability engineering", "sre",

    # Data Science & Machine Learning
    "machine learning", "deep learning", "artificial intelligence",
    "natural language processing", "nlp", "computer vision",
    "data science", "data analysis", "data engineering",
    "pandas", "numpy", "scipy", "scikit-learn",
    "tensorflow", "keras", "pytorch",
    "xgboost", "lightgbm",
    "opencv",
    "statistics", "linear regression", "logistic regression",
    "clustering", "classification", "time series",

    # Big Data
    "hadoop", "spark", "pyspark", "kafka", "flink",
    "hive", "pig", "hbase", "airflow",
    "data warehousing", "etl",

    # Mobile Development
    "android", "ios",
    "react native", "flutter", "xamarin",
    "android studio", "xcode",

    # Cybersecurity
    "cybersecurity", "information security",
    "penetration testing", "ethical hacking",
    "network security", "application security",
    "cryptography", "siem", "soc",
    "firewalls", "ids", "ips",
    "owasp", "iam",

    # Networking
    "tcp/ip", "udp", "dns", "dhcp",
    "http", "https",
    "routing", "switching",
    "vpn", "lan", "wan",
    "ccna", "ccnp",

    # Operating Systems
    "windows", "linux", "macos",
    "red hat", "ubuntu", "debian", "centos",

    # Software Engineering
    "object oriented programming", "oop",
    "design patterns",
    "clean code", "solid principles",
    "data structures", "algorithms",
    "microservices", "monolithic architecture",
    "event driven architecture",

    # Testing & QA
    "unit testing", "integration testing", "system testing",
    "selenium", "cypress", "playwright",
    "junit", "pytest", "testng",
    "automation testing", "manual testing",

    # Version Control & Tools
    "git", "github", "gitlab", "bitbucket",
    "jira", "confluence",
    "postman", "swagger",

    # UI / UX
    "figma", "adobe xd", "sketch",
    "ui design", "ux design",
    "wireframing", "prototyping",

    # ERP / CRM / Enterprise
    "sap", "oracle erp", "salesforce",
    "workday", "servicenow",

    # Methodologies
    "agile", "scrum", "kanban",
    "waterfall", "devsecops",

    # Misc / Emerging
    "blockchain", "web3", "smart contracts",
    "solidity",
    "internet of things", "iot",
    "robotic process automation", "rpa",
    "computer graphics", "game development",
    "unity", "unreal engine"
]

import re

def auto_label(text, skills):
    entities = []
    text_lower = text.lower()
    for skill in skills:
        for match in re.finditer(rf"\b{re.escape(skill)}\b", text_lower):
            entities.append((match.start(), match.end(), "SKILL"))
    return (text, {"entities": entities})

descriptions = df["description"].tolist()
TRAIN_DATA = [auto_label(desc, SKILLS) for desc in descriptions]

from sklearn.model_selection import train_test_split
train_data, dev_data = train_test_split(
    TRAIN_DATA,
    test_size=0.2,
    random_state=42
)

In [14]:
import spacy
from spacy.tokens import DocBin
from spacy.util import filter_spans


nlp = spacy.blank("en")
ner = nlp.add_pipe("ner")
ner.add_label("SKILL")

train_db = DocBin()
dev_db = DocBin()


for text, annotations in train_data:
    doc = nlp.make_doc(text)
    ents = []
    for start, end, label in annotations["entities"]:
        span = doc.char_span(start, end, label=label)
        if span:
            ents.append(span)
    doc.ents = filter_spans(ents)
    train_db.add(doc)


for text, annotations in dev_data:
    doc = nlp.make_doc(text)
    ents = []
    for start, end, label in annotations["entities"]:
        span = doc.char_span(start, end, label=label)
        if span:
            ents.append(span)
    doc.ents = filter_spans(ents)
    dev_db.add(doc)

train_db.to_disk("train.spacy")
dev_db.to_disk("dev.spacy")


!python -m spacy init config config.cfg --lang en --pipeline transformer,ner --force
!python -m spacy train config.cfg --output ./output --paths.train train.spacy --paths.dev dev.spacy

ℹ Generated config template specific for your use case
- Language: en
- Pipeline: ner
- Optimize for: efficiency
- Hardware: CPU
- Transformer: None
✔ Auto-filled config with all values
✔ Saved config
config.cfg
You can now add your data and train your pipeline:
python -m spacy train config.cfg --paths.train ./train.spacy --paths.dev ./dev.spacy
✔ Created output directory: output
ℹ Saving to output directory: output
ℹ Using CPU

=========================== Initializing pipeline ===========================
✔ Initialized pipeline

============================= Training pipeline =============================
ℹ Pipeline: ['tok2vec', 'ner']
ℹ Initial learn rate: 0.001
E    #       LOSS TOK2VEC  LOSS NER  ENTS_F  ENTS_P  ENTS_R  SCORE 
---  ------  ------------  --------  ------  ------  ------  ------
  0       0          0.00    141.00    0.22    0.14    0.53    0.00
  0     200         77.16   5351.71   90.16   99.36   82.51    0.90
  0     400        112.09    337.40   95.35   96.74   93

In [15]:
import spacy

nlp = spacy.load("./output/model-best")

def extract_skills(text):
    doc = nlp(text)
    return list(set(
        ent.text.lower()
        for ent in doc.ents
        if ent.label_ == "SKILL"
    ))

df["extracted_skills"] = df["description"].apply(extract_skills)


In [16]:
import pandas as pd


skills_long = df.explode("extracted_skills")


skills_long = skills_long.rename(columns={"extracted_skills": "skill"})

skills_long = skills_long.dropna(subset=["skill"])


role_skill_counts = (
    skills_long
    .groupby(["normalized_role", "skill"])
    .size()
    .reset_index(name="count")
)


role_skill_counts["rank"] = (
    role_skill_counts
    .groupby("normalized_role")["count"]
    .rank(method="first", ascending=False)
)


top_skills = role_skill_counts[role_skill_counts["rank"] <= 10]


top_skills = top_skills.sort_values(
    ["normalized_role", "rank"]
)


top_skills.to_csv("ranked_skills.csv", index=False)

print("CSV successfully created.")

CSV successfully created.


Spacy Output

In [17]:

import pandas as pd

# Explode the list of skills
# [python, sql]
df_exploded = df.explode('extracted_skills')

# Filter out empty/none
df_exploded = df_exploded.dropna(subset=['extracted_skills'])
df_exploded = df_exploded[df_exploded['extracted_skills'] != '']

# Count skills
skill_counts = df_exploded.groupby(['normalized_role', 'extracted_skills']).size().reset_index(name='count')

# (frequency %)
role_totals = df['normalized_role'].value_counts().reset_index()
role_totals.columns = ['normalized_role', 'total_postings']

# calculate the score
skill_ranking = skill_counts.merge(role_totals, on='normalized_role')
skill_ranking['score'] = skill_ranking['count'] / skill_ranking['total_postings']

# Sort by role
skill_ranking = skill_ranking.sort_values(by=['normalized_role', 'score'], ascending=[True, False])

desired_roles = [
    "software engineer",
    "qa engineer",
    "cybersecurity engineer",
    "data engineer",
    "data scientist",
    "database administrator",
    "devops engineer",
    "embedded engineer",
    "data analyst"
]


for role in desired_roles:
    role_data = skill_ranking[skill_ranking['normalized_role'] == role]

    if not role_data.empty:
        print(f"\nSkills for {role}:")
        top_skills = role_data[['extracted_skills', 'score']].head(10)
        print(top_skills.rename(columns={'extracted_skills': 'skill'}).to_string(index=False))
    else:
        print(f"\nSkills for {role}: No data found.")


Skills for software engineer:
     skill    score
    python 0.446880
javascript 0.386172
       aws 0.365936
     react 0.357504
        go 0.261383
      java 0.254637
     agile 0.219224
kubernetes 0.204047
       sql 0.193929
    docker 0.190556

Skills for qa engineer:
     skill    score
   postman 0.571429
     agile 0.500000
  selenium 0.500000
   cypress 0.428571
javascript 0.428571
      java 0.357143
      jira 0.357143
    devops 0.285714
     scrum 0.285714
       sql 0.285714

Skills for cybersecurity engineer:
               skill    score
                  go 0.666667
          javascript 0.666667
              python 0.666667
               agile 0.333333
application security 0.333333
                 aws 0.333333
     computer vision 0.333333
     design patterns 0.333333
           devsecops 0.333333
              docker 0.333333

Skills for data engineer:
           skill    score
data engineering 1.000000
          python 1.000000
             sql 1.000000
       